In [1]:
import pandas as pd
import numpy as np
import sklearn
import statsmodels.api as sm
import os

In [2]:
database = pd.read_excel("data/DataBase.xlsx")
dataset = pd.read_excel("data/DataSigns.xlsx")

In [3]:
print(database.head(1000))

                               ФИО    №  Возраст  Пол  Креат. 1  рСКФ  \
0      Авсеевич Татьяна Васильевна    1       84    2     80.30    58   
1                     Алишоев С.М.    2       46    1     92.22    85   
2        Андреева Галина Михайлова    3       65    2     67.10    83   
3      Артемьев Евгений Евгеньевич    4       79    1    126.50    46   
4     Астахов Александр Николаевич    5       65    1    109.00    61   
..                             ...  ...      ...  ...       ...   ...   
300      Федорова Ольга Лукьяновна  301       67    2     78.50    67   
301   Фомичева Антонина Михайловна  302       67    2     88.94    58   
302  Харсеева Наталия Вячеславовна  303       45    2     76.10    82   
303   Хорькова Татьяна Григорьевна  304       65    2     91.08    57   
304     Шулев Владимир Геннадьевич  305       55    1    111.70    64   

     Критерий Х  Х_Оценка                                            Диагноз  \
0            53        СТ  внебольн.двустор

In [4]:
def clinic_value(row):
    if row == "СТ":
        return 1
    elif row == "Т":
        return 2
    elif row == "Л":
        return 0
    

## Переведем клиническую оценку в числовой аналог.
## Сonvert the clinical assessment into a numerical equivalent.

In [5]:
database["Result_Value"] = database["Клин_Оценка"].apply(clinic_value)

print(database.head(1000))

                               ФИО    №  Возраст  Пол  Креат. 1  рСКФ  \
0      Авсеевич Татьяна Васильевна    1       84    2     80.30    58   
1                     Алишоев С.М.    2       46    1     92.22    85   
2        Андреева Галина Михайлова    3       65    2     67.10    83   
3      Артемьев Евгений Евгеньевич    4       79    1    126.50    46   
4     Астахов Александр Николаевич    5       65    1    109.00    61   
..                             ...  ...      ...  ...       ...   ...   
300      Федорова Ольга Лукьяновна  301       67    2     78.50    67   
301   Фомичева Антонина Михайловна  302       67    2     88.94    58   
302  Харсеева Наталия Вячеславовна  303       45    2     76.10    82   
303   Хорькова Татьяна Григорьевна  304       65    2     91.08    57   
304     Шулев Владимир Геннадьевич  305       55    1    111.70    64   

     Критерий Х  Х_Оценка                                            Диагноз  \
0            53        СТ  внебольн.двустор

## Выведем таблицу признаков для каждого пациента.

## Create a table of symptoms for each patient.

In [6]:
print(dataset.head(1000))

     Age  Temperature.  SatO2  Neutral.1  Ctreating. 1     SRB  fibrinogen  \
0     84          37.0   92.0       87.5         80.30   17.00        9.08   
1     46          38.0   97.0       71.2         92.22   70.00        4.48   
2     65          37.7   97.0       51.7         67.10   15.00        4.40   
3     79          37.2   93.0       67.9        126.50   37.00        5.52   
4     65          37.0   80.0       82.4        109.00  155.00        5.97   
..   ...           ...    ...        ...           ...     ...         ...   
300   67          36.4   98.0       55.7         78.50   34.22        6.11   
301   67          37.0   97.0       69.8         88.94   42.79        6.11   
302   45          37.0   97.0       79.9         76.10  164.98        8.39   
303   65          37.1   99.0       55.0         91.08    6.70        4.60   
304   55          39.7   97.0       64.7        111.70  111.60        9.40   

       №   X  Xcontinuous  
0      1  53      53.0250  
1      

## Матрица корреляции для независимых величин. Используем метод Спирмана, т.к результирующая матрица будет более устойчива к выбросам, нежели простая матрица корреляции.

## Correlation matrix for independent variables. We use the Spearman method, as the resulting matrix will be more resistant to outliers than a regular correlation matrix.

In [7]:
dataset.corr(method='spearman')

,Age,Temperature.,SatO2,Neutral.1,Ctreating. 1,SRB,fibrinogen,№,X,Xcontinuous
Age,1.000000,-0.120748,-0.101964,0.221996,0.093304,0.156309,-0.023994,-0.111286,0.342135,0.344044
Temperature.,-0.120748,1.000000,-0.083280,0.079555,0.089971,0.193079,0.050887,-0.051112,0.146029,0.145166
SatO2,-0.101964,-0.083280,1.000000,-0.192370,-0.055251,-0.236870,-0.158817,0.134208,-0.229921,-0.229943
Neutral.1,0.221996,0.079555,-0.192370,1.000000,0.096871,0.456748,0.316116,-0.023758,0.563685,0.563977
Ctreating. 1,0.093304,0.089971,-0.055251,0.096871,1.000000,0.127338,0.093101,-0.029828,0.361593,0.359291
SRB,0.156309,0.193079,-0.236870,0.456748,0.127338,1.000000,0.510256,0.070656,0.927619,0.927744
fibrinogen,-0.023994,0.050887,-0.158817,0.316116,0.093101,0.510256,1.000000,0.028751,0.491219,0.492519
№,-0.111286,-0.051112,0.134208,-0.023758,-0.029828,0.070656,0.028751,1.000000,0.031128,0.034642
X,0.342135,0.146029,-0.229921,0.563685,0.361593,0.927619,0.491219,0.031128,1.000000,0.999643
Xcontinuous,0.344044,0.145166,-0.229943,0.563977,0.359291,0.927744,0.492519,0.034642,0.999643,1.000000


## Подготовим данные для регрессии. Оценим влияние семи независимых параметров, а так же свободного члена на числовую оценку Result_Value.

## Prepare the data for regression. We will assess the impact of seven independent parameters, as well as the constant term, on the numerical estimate of Result_Value.

### $$y = \beta_0 + \beta_1 Age + \beta_2 Temperature + \dots + \beta_7 Fibrinogen$$

In [8]:
features = ['Age', 'Temperature.', 'SatO2', 'Neutral.1', 'Ctreating. 1', 'SRB', 'fibrinogen']

X = dataset[features].fillna(dataset[features].mean())

X = sm.add_constant(X)

y = database["Result_Value"]


## Разобьём данные в стандартном соотношении 70 на 30. Будем проводить обучение на 70 процентах данных и тестировать модель на оставшихся 30.

## Split the data in the standard 70-to-30 ratio. We’ll train the model on 70 percent of the data and test it on the remaining 30 percent.

In [9]:
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size=0.30, random_state=100)

model = sm.OLS(y_train, X_train)

results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:           Result_Value   R-squared:                       0.126
Model:                            OLS   Adj. R-squared:                  0.096
Method:                 Least Squares   F-statistic:                     4.228
Date:                Mon, 17 Aug 2026   Prob (F-statistic):           0.000222
Time:                        19:23:44   Log-Likelihood:                -101.77
No. Observations:                 213   AIC:                             219.5
Df Residuals:                     205   BIC:                             246.4
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.3902      1.557      0.251   

In [10]:
y_pred_train_continuous = results.predict(X_train)

y_pred_test_continuous = results.predict(X_test)

mae_train = sklearn.metrics.mean_absolute_error(y_train, y_pred_train_continuous)

r2_train = sklearn.metrics.r2_score(y_train, y_pred_train_continuous)

mae_test = sklearn.metrics.mean_absolute_error(y_test, y_pred_test_continuous)

r2_test = sklearn.metrics.r2_score(y_test, y_pred_test_continuous)

print(f"Для обучения: {mae_train}, {r2_train}", end="\n")

print(f"Для экзамена: {mae_test}, {r2_test}", end="")

Для обучения: 0.2923089927743731, 0.12616731359426026
Для экзамена: 0.28657827577152767, 0.032903366521185085

In [11]:
y_pred_train_rounded = np.clip(np.round(y_pred_train_continuous), 0, 2)

y_pred_test_rounded = np.clip(np.round(y_pred_test_continuous), 0, 2)

mae_train = sklearn.metrics.mean_absolute_error(y_train, y_pred_train_rounded)

r2_train = sklearn.metrics.r2_score(y_train, y_pred_train_rounded)

mae_test = sklearn.metrics.mean_absolute_error(y_test, y_pred_test_rounded)

r2_test = sklearn.metrics.r2_score(y_test, y_pred_test_rounded)

print(f"Для обучения: {mae_train}, {r2_train}", end="\n")

print(f"Для экзамена: {mae_test}, {r2_test}", end="")

Для обучения: 0.22535211267605634, -0.29352226720647767
Для экзамена: 0.17391304347826086, -0.07288629737609331

In [12]:
acc_train = sklearn.metrics.accuracy_score(y_train, y_pred_train_rounded)
acc_test = sklearn.metrics.accuracy_score(y_test, y_pred_test_rounded)
print("--- Точность предсказаний ---")
print(f"На обучающей выборке: {acc_train * 100:.1f}%")
print(f"На тестовой выборке: {acc_test * 100:.1f}%")

--- Точность предсказаний ---
На обучающей выборке: 77.5%
На тестовой выборке: 82.6%
